# 第22章 分组、聚合与数据透视

使用groupby、agg、transform、pivot_table和crosstab回答分组问题。

## 本章定位

本章按“概念 → 示例 → 练习”的顺序组织。代码单元格可以单独运行，也可以从上到下完整运行。

## 学习目标

- 执行分组聚合
- 一次计算多个指标
- 保留原行的组内计算
- 构建透视表和交叉表


## 核心概念

- 分组前必须明确维度、指标和聚合函数。
- agg压缩行数，transform保持原行数。
- 透视表中的缺失组合与真实0含义不同。


## 示例 1：分组聚合

命名聚合让输出列直接表达口径。


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "region": ["华东", "华东", "华南", "华南", "华北", "华北"],
    "channel": ["线上", "线下", "线上", "线下", "线上", "线下"],
    "amount": [520, 310, 460, 280, 390, 260],
    "quantity": [3, 2, 2, 1, 2, 1],
})
summary = orders.groupby("region").agg(
    sales=("amount", "sum"),
    orders=("amount", "size"),
    average=("amount", "mean"),
)
print(summary)


## 示例 2：transform组内占比

transform结果与原表等长，可直接添加为新列。


In [ ]:
orders["region_total"] = orders.groupby("region")["amount"].transform("sum")
orders["region_share"] = orders["amount"] / orders["region_total"]
print(orders)


## 示例 3：透视表与交叉表

index与columns分别定义行维度和列维度。


In [ ]:
pivot = orders.pivot_table(
    index="region", columns="channel", values="amount", aggfunc="sum", fill_value=0
)
counts = pd.crosstab(orders["region"], orders["channel"], margins=True)
print(pivot)
print(counts)


## 初学者学习路线

这章建议按照“先观察、再模仿、后修改、最后独立完成”的顺序学习，不必一次记住所有参数。

1. 先阅读任务说明，明确这段代码要回答什么问题。
2. 运行一个最小例子，先观察输入、输出和数据形状，再回看每一行代码。
3. 只修改一个参数或一条数据，重新运行并比较前后结果。
4. 完成“综合练习”，最后再看本章小结，把能迁移到其他数据的问题写下来。

运行时如果看到 NameError，通常是前置单元格还没有运行；如果输出和预期不同，先检查变量是否被后面的单元格重新赋值。


## 先做一个小检查

进入正式例子前，先用一句话回答：本章的输入是什么，想得到什么结果？

本章主题是“第22章 分组、聚合与数据透视”。请特别留意三件事：输入的类型或形状、处理中间变量的含义、最后输出能否支持一个清楚的结论。


## Pandas 的学习主线

写数据字典 → 读取与检查 → 清洗类型和缺失值 → 选择与筛选 → 新增计算列 → 分组聚合 → 合并或透视 → 导出可复用结果

每一步都说明“一行代表什么”。处理前后记录行数、列数和关键字段；汇总前先确认分组粒度，避免得到数字却无法解释。


## 本模块练习方式

基础：完成一个字段清洗；提高：从明细表生成汇总表；挑战：处理重复、缺失和类型混乱，并写出清洗规则。

完成后请写下：输入是什么、处理做了什么、输出说明了什么、还存在什么限制。


## 示例 4：从明细表生成计算列

这一组例子只处理一个小问题。先运行代码，再逐行对照拆解说明。


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "region": ["华东", "华南", "华东"],
    "sales": [120, 150, 180],
    "cost": [80, 100, 130],
})
orders["profit"] = orders["sales"] - orders["cost"]
orders["profit_rate"] = orders["profit"] / orders["sales"]
print(orders.round(3))


### 逐步拆解

先新增一个简单指标，再基于它计算比例；拆成多列可以保留中间结果并方便检查。

建议第一次运行后只改一个输入值，再观察哪一个输出发生变化。


## 示例 5：从明细汇总到业务表

看懂上一个例子后，再观察同一主题在另一种数据或场景中的写法。


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "region": ["华东", "华东", "华南", "华南"],
    "channel": ["线上", "线下", "线上", "线下"],
    "sales": [120, 80, 150, 100],
})
summary = orders.groupby(["region", "channel"], as_index=False)["sales"].sum()
print(summary)
print("地区合计：")
print(orders.groupby("region")["sales"].sum())


### 逐步拆解

先明确每一行的粒度，再选择 groupby 的字段；汇总表的每一行代表一个清晰的分组组合。

自我检查：如果把输入数量、类别或参数改成另一组值，代码是否仍然能运行？


## 教学实验：分组汇总与粒度

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "region": ["华东", "华东", "华南", "华南"],
    "channel": ["线上", "线下", "线上", "线下"],
    "sales": [120, 80, 150, 100],
})
summary = orders.groupby("region", as_index=False)["sales"].sum()
print(summary)
print("汇总表每一行代表一个地区")


### 第一个结果怎么读

先确认明细表一行代表一笔订单，再确认汇总表一行代表一个地区。`groupby` 的字段决定结果的粒度。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
orders["sales_level"] = orders["sales"].map(
    lambda value: "高" if value >= 120 else "普通"
)
print(orders)
print(orders["sales_level"].value_counts())


### 第二个结果怎么读

第二个实验只增加一个分类列，不改变原始销售额。练习解释：什么时候应该新增列，什么时候应该直接筛选行？

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：脏数据转换怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

raw = pd.Series(["12", "unknown", "18", ""])
converted = pd.to_numeric(raw, errors="coerce")
print("转换结果：")
print(converted)
print("无法转换的数量：", converted.isna().sum())
print("后续可以选择删除、填充或回查原始值。")


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

errors="coerce" 会把无法转换的值记录为缺失，适合先完成质量盘点；不要在没有统计数量前直接删除。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 常见误区

- 使用mean却把结果描述为合计
- groupby后忘记处理索引
- 把缺失组合无条件填0


## 综合练习

1. 按地区和渠道分组
2. 计算销售额、订单数和平均订单金额
3. 构建地区×渠道透视表

请先独立完成，再点击下方“显示答案”查看参考代码。


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "region": ["华东", "华东", "华南", "华南", "华北"],
    "channel": ["广告", "自然", "广告", "自然", "广告"],
    "amount": [620, 410, 530, 380, 470],
})

# TODO: 按地区和渠道分组，计算销售额、订单数和平均订单金额
summary = orders.groupby(["region", "channel"])["amount"].agg(["sum", "size", "mean"]).reset_index()
summary.columns = ["region", "channel", "sales", "order_count", "average_order"]

# TODO: 构建地区×渠道透视表（销售额）
pivot = orders.pivot_table(index="region", columns="channel", values="amount", aggfunc="sum", fill_value=0)

print(summary)
print(pivot)


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "region": ["华东", "华东", "华南", "华南", "华北"],
    "channel": ["广告", "自然", "广告", "自然", "广告"],
    "amount": [620, 410, 530, 380, 470],
})
summary = orders.groupby(["region", "channel"]).agg(
    sales=("amount", "sum"),
    order_count=("amount", "size"),
    average_order=("amount", "mean"),
).reset_index()
pivot = summary.pivot(index="region", columns="channel", values="sales").fillna(0)
print(summary)
print(pivot)

# 自检
assert len(summary) == 4, "检查分组结果：应该有4个组合"
assert pivot.loc["华东", "广告"] == 620, "检查透视表：华东-广告应该是620"


## 本章小结

使用groupby、agg、transform、pivot_table和crosstab回答分组问题。

**迁移思考**：

1. 如果需要计算每个地区销售额占全国的比例，应该用 agg 还是 transform？为什么？
2. 为什么透视表中的缺失组合不能无条件填0？什么情况下填0是合理的？


### 你已经掌握

- 执行分组聚合
- 一次计算多个指标
- 保留原行的组内计算
- 构建透视表和交叉表


### 关键知识速查

| 知识点 | 作用与提醒 | 关键写法 |
| --- | --- | --- |
| 分组聚合 | 命名聚合让输出列直接表达口径。 | `pd.DataFrame()`、`orders.groupby()`、`.agg()` |
| transform组内占比 | transform结果与原表等长，可直接添加为新列。 | `orders.groupby()`、`.transform()`、`orders["region_total"]`、`orders["region_share"]` |
| 透视表与交叉表 | index与columns分别定义行维度和列维度。 | `orders.pivot_table()`、`pd.crosstab()`、`orders["region"]`、`orders["channel"]` |


### 需要注意

- 使用mean却把结果描述为合计
- groupby后忘记处理索引
- 把缺失组合无条件填0


### 完成检查

- [ ] 能够执行分组聚合
- [ ] 能够一次计算多个指标
- [ ] 能够保留原行的组内计算
- [ ] 能够构建透视表和交叉表
